# Denoise with NGMeet

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from EDX import *
from utils import *
from utils_sofima import *
import torch
from torch.utils.data import DataLoader
from model import UNet
from utils_noise import *
from bm3d import bm3d, BM3DStages
from bm4d import bm4d, BM4DStages
import pickle
import copy
import time
import humanfriendly
from skimage.restoration import estimate_sigma
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap, LinearSegmentedColormap


device = torch.device("mps")
print(device)

%load_ext autoreload
%autoreload 2

### Load aligned tile

In [ ]:
with open('../preprocessing_basic/results/preprocessed_edx/20251201_142554_tile_aligned.pkl', 'rb') as file:
    tile = pickle.load(file)

tile.apply("crop", parameters = {"crop_idx": (slice(50,1024),slice(50,1024),slice(None,None,None))})
h, w, b = tile.EDX_dim
tile.summary()

### NGMeet in matlab

In [ ]:
# Start matlab engine and set path
eng = matlab.engine.start_matlab()
matlab_path = eng.genpath('../matlab/')   # add path recursively
eng.addpath(matlab_path, nargout=0)

In [ ]:
# 
hsi_clean = np.ascontiguousarray(tile.EDX)
print(hsi_clean.shape, hsi_clean.dtype)
EDX_clean = eng.denoiseNGMeet(hsi_clean)